# Off-axis hologram of a magnetic cube from summed Gaussian beamlets

This notebook simulates an electron hologram in the example microscope (`microscope_models/microscope.toml`) with the
objective lens on — **not** Lorentz mode — by propagating a few hundred thousand coherent Gaussian beamlets
through the column and summing their complex fields on the detector.

| Plane | Component | What each Gaussian receives |
|---|---|---|
| Specimen | `InterpolatedSample2D` from the projected magnetic cube | local optical path + log-amplitude, gradients, Hessians |
| Objective `Obj_post` | `ElectromagneticLens` (focus + Larmor rotation) | exact quadratic update |
| Back focal plane | `PupilAberration` (defined below): C10, C12, C30 | on-axis aberration phase χ(θ) |
| Biprism | `PhaseBiprism`: ±β deflection **and** an opaque wire | complex action `path − i·log T / k` |
| `IL1`, `IL2`, `IL3` | `ElectromagneticLens`, refocused for 150 000× | exact quadratic update |
| `IL3` image | `ImageDistortion` (defined below): projector radial + spiral distortion | canonical point map |
| Projector `PL1` | `ElectromagneticLens` | exact quadratic update |
| Detector | `Detector` | coherent sum with `evaluate_gaussians` |

The biprism wire is an amplitude aperture. Its transmission enters each beamlet through the imaginary part of the
complex action, which shifts, attenuates and reshapes the Gaussian. Projector distortion acts after the biprism,
so it bends the interference fringes as well as the image of the cube.

In [ ]:
import os
os.environ["JAX_ENABLE_X64"] = "1"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import sys
import time
from dataclasses import replace
from pathlib import Path

import jax
import jax.numpy as jnp
import jax_dataclasses as jdc
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from scipy.ndimage import gaussian_filter
from scipy.optimize import brentq

REPO = Path.cwd().resolve().parents[1]
# The magnetic-cube recipe is shared with the archived holography notebooks.
sys.path.insert(0, str(REPO / "archived" / "examples" / "holography_distortion"))
from magnetic_cube import make_tilted_uniform_cube, project_phase_along_z

from temgym_core.components import (
    Detector, GaussianActionComponent,
    InterpolatedSample2D, Lens, PhaseBiprism, Plane,
)
from temgym_core.constants import energy2wavelength
from temgym_core.evaluate import evaluate_gaussians
from temgym_core.microscope import Microscope, find_imaging_condition
from temgym_core.ray import Ray
from temgym_core.run import column_matrix, run_to_end
from temgym_core.source import rectangular_input_wave

OUTPUT = Path("output")
OUTPUT.mkdir(exist_ok=True)
device = jax.devices()[0]
print(device.device_kind, "| JAX", jax.__version__)

## The microscope at 150 000×

Load the example microscope, take the 150k row of its magnification table and refocus `IL1`–`IL3` with
`find_imaging_condition` (A = 150 000 × I, B = 0, zero net rotation). The objective and `PL1` currents are fixed.
From the native lenses we locate the objective's back focal plane (A = 0), its first image (B = 0), the image
formed by `IL3` (the projector's object plane) and the Larmor rotation of the first image.

In [ ]:
M_total = 150_000
microscope = Microscope.from_toml(REPO / "examples" / "microscope_models" / "microscope.toml")
microscope = find_imaging_condition(microscope.with_settings(magnification=M_total),
                                    magnification=M_total, vary=["IL1", "IL2", "IL3"]).microscope
voltage = microscope.voltage
wavelength = float(energy2wavelength(voltage))
k = 2 * np.pi / wavelength
lenses = {name: microscope.components[name] for name in ("Obj_post", "IL1", "IL2", "IL3", "PL1")}
z_sample = float(microscope.components["sample"].z)


def matrix_to(z_end, last=np.inf):
    # Native 5x5 matrix from the specimen to z_end through lenses up to z = last.
    column = tuple(c for c in microscope.column if z_sample < float(c.z) < z_end and float(c.z) <= last)
    return np.asarray(column_matrix(column + (Plane(z=z_end),), source_z=z_sample, voltage=voltage))


z_obj = float(lenses["Obj_post"].z)
z_bfp = brentq(lambda z: matrix_to(z, z_obj)[0, 0], z_obj + 1e-4, z_obj + 0.02)
z_img = brentq(lambda z: matrix_to(z, z_obj)[0, 2], z_obj + 0.05, float(lenses["IL1"].z) - 1e-3)
z_il3_image = brentq(lambda z: matrix_to(z, float(lenses["IL3"].z))[0, 2],
                     float(lenses["IL3"].z) + 1e-3, float(lenses["PL1"].z) - 1e-6)
first_image = matrix_to(z_img, z_obj)[:2, :2]
M_obj = np.sqrt(abs(np.linalg.det(first_image)))
image_angle = np.arctan2(first_image[1, 0], first_image[0, 0])    # direction of specimen +x in the first image
total = matrix_to(float(microscope.components["detector"].z))

print(f"{voltage/1e3:.0f} kV, λ = {wavelength*1e12:.3f} pm")
for name, lens in lenses.items():
    print(f"  {name:8s} z = {float(lens.z)*1e3:8.3f} mm   f = {float(lens.focal_length(voltage))*1e3:7.3f} mm   "
          f"I = {float(lens.current):+.4f} A   rotation {np.rad2deg(float(lens.rotation_angle(voltage))):+8.2f}°")
print(f"specimen {z_sample*1e3:.3f} mm, objective BFP {z_bfp*1e3:.3f} mm, first image {z_img*1e3:.3f} mm "
      f"(|M| = {M_obj:.2f}, specimen +x at {np.rad2deg(image_angle):.1f}°), IL3 image {z_il3_image*1e3:.3f} mm")
print(f"specimen → detector: A = {total[0,0]:.0f}, {total[0,1]:.1e};  max |B| = {np.abs(total[:2,2:4]).max():.1e} m")

## Biprism geometry

The biprism sits b = 100 mm above the first image, with its wire along the image of the specimen y axis. With
deflection β per side, the fringe spacing in the first image is λ/(2β), each half-image moves by βb, and the
wire's shadow is magnified from the back-focal-plane crossover. `Microscope.from_toml` supplies a 256² Medipix
with 55 µm pixels; at 150 000× that gives about one pixel per fringe, so this notebook records on a 1024² camera
with 14 µm pixels at the same position.

A Gaussian launched at the specimen with waist w is at least √((A w)² + (λB/πw)²) wide at the wire, where A and B
are the specimen-to-wire matrix elements. The waist below is close to the one that minimises this width.

In [ ]:
b = 100e-3
z_bip = z_img - b
beta = 3.5e-5                      # rad deflection per side
wire = 0.6e-6                      # wire diameter
waist, lattice, illuminated = 3e-9, 1.5e-9, 600e-9

L = z_img - z_bfp
shadow = wire * L / (z_bip - z_bfp)
A_wire = M_obj * (z_bip - z_bfp) / L
B_wire = b / M_obj
fresnel = np.sqrt(wavelength * b * (z_bip - z_bfp) / L)
width_at_wire = np.hypot(A_wire * waist, wavelength * B_wire / (np.pi * waist))
print(f"biprism z = {z_bip*1e3:.1f} mm; fringe spacing {wavelength/(2*beta)*1e9:.1f} nm in the first image, "
      f"{wavelength/(2*beta*M_obj)*1e9:.2f} nm at the specimen, {wavelength/(2*beta)*M_total/M_obj*1e6:.0f} µm on the camera")
print(f"half-image shift βb = {beta*b*1e6:.2f} µm, wire shadow {shadow*1e6:.2f} µm, "
      f"overlap {(2*beta*b - shadow)*1e6:.2f} µm = {(2*beta*b - shadow)/M_obj*1e9:.0f} nm at the specimen")
print(f"specimen beamlet at the wire: {width_at_wire*1e6:.2f} µm (optimum waist "
      f"{np.sqrt(wavelength*B_wire/(np.pi*A_wire))*1e9:.1f} nm); Fresnel scale √(λb') = {fresnel*1e6:.2f} µm")

detector = Detector(z=float(microscope.components["detector"].z), pixel_size=(14e-6, 14e-6), shape=(1024, 1024))
print(f"camera field of view at the specimen: {1024*14e-6/M_total*1e9:.0f} nm")

## Magnetic cube specimen

A 40 nm cube, tilted by 20° about x, carries a 20 V mean inner potential and a uniform 1 T induction; field
amplitude falls to 0.8 across its thickness. `project_phase_along_z` integrates the electrostatic and vector
potentials along the beam. The maps are smoothed by a 1.5 nm Gaussian so that the local quadratic expansion applied
to each 3 nm beamlet sees a resolved edge, then converted to optical path φ/k and log amplitude. The cube is centred
βb/|M| to the left of the axis so that the biprism overlaps its image with vacuum from the right.

In [ ]:
cube_side, map_width = 40e-9, 160e-9
cube_x = -beta * b / M_obj
fields, xs, ys, zs = make_tilted_uniform_cube(
    tilt_x=np.deg2rad(20), N=256, L=map_width, cube_fraction=cube_side / map_width,
    V0=20.0 * cube_side,          # the recipe divides V0 by the cube side: 20 V mean inner potential
    B0=1.0, amplitude_dampening=0.8, x0=cube_x,
)
amplitude_xy, phase_xy = project_phase_along_z(fields, zs, voltage)
xs, ys = np.asarray(xs), np.asarray(ys)
smooth = 1.5e-9 / (xs[1] - xs[0])
phase_map = gaussian_filter(np.asarray(phase_xy).T, smooth)
log_amplitude_map = gaussian_filter(np.log(np.asarray(amplitude_xy).T), smooth)

sample = InterpolatedSample2D.from_action_maps(
    pathlength_yx_m=phase_map / k, log_amplitude_yx=log_amplitude_map,
    x_coords_m=xs, y_coords_m=ys, z=z_sample,
)
print(f"cube centre x = {cube_x*1e9:.1f} nm; projected phase 0 … {phase_map.max():.2f} rad, "
      f"minimum amplitude {np.exp(log_amplitude_map.min()):.2f}")

fig, axes = plt.subplots(1, 2, figsize=(9, 4), constrained_layout=True)
extent_nm = np.array([xs[0], xs[-1], ys[0], ys[-1]]) * 1e9
for ax, image, title, cmap in ((axes[0], phase_map, "projected phase (rad)", "twilight"),
                              (axes[1], np.exp(log_amplitude_map), "amplitude", "gray")):
    plt.colorbar(ax.imshow(image, extent=extent_nm, origin="lower", cmap=cmap), ax=ax, shrink=0.85)
    ax.set(title=title, xlabel="x (nm)", ylabel="y (nm)")

## The specimen as a sum of Gaussians

The illumination is a square lattice of 3 nm Gaussians on a 1.5 nm pitch, covering 600 nm so that the edge of the
illuminated area stays well outside the recorded field. Applying the specimen gives each beamlet the local value,
slope and curvature of the complex action, so the bundle leaving the specimen *is* the Gaussian interpolation of
the exit wave. The cell compares that sum with the smoothed map.

In [ ]:
beam = rectangular_input_wave(aperture_width=illuminated, aperture_height=illuminated, waist=waist,
                              voltage=voltage, overlap_factor=waist / lattice, z0=z_sample)
n_specimen = beam.x.shape[0]
propagate = jax.jit(run_to_end)
exit_wave = propagate(beam, (sample,))

check_grid = Detector(z=z_sample, pixel_size=(0.5e-9, 0.5e-9), shape=(256, 256), centre=(cube_x, 0.0))
gaussian_exit = np.asarray(evaluate_gaussians(exit_wave, check_grid, method="spatial"))
gx, gy = (np.asarray(c) for c in check_grid.coords_1d)
target = np.asarray(sample.evaluate_complex(jnp.stack(jnp.meshgrid(gx, gy), -1), wavelength_m=wavelength))
error = np.linalg.norm(gaussian_exit - target) / np.linalg.norm(target)
print(f"{n_specimen:,} specimen Gaussians; exit-wave relative error over the cube region: {error:.2%}")

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), constrained_layout=True)
ext = np.array([gx[0], gx[-1], gy[-1], gy[0]]) * 1e9
axes[0].imshow(np.angle(target), extent=ext, cmap="twilight"); axes[0].set_title("map: phase")
axes[1].imshow(np.angle(gaussian_exit), extent=ext, cmap="twilight"); axes[1].set_title("sum of Gaussians: phase")
im = axes[2].imshow(np.abs(gaussian_exit - target), extent=ext, cmap="magma"); axes[2].set_title("|difference|")
plt.colorbar(im, ax=axes[2], shrink=0.85)
for ax in axes: ax.set(xlabel="x (nm)", ylabel="y (nm)")

## Aberrations, biprism and projector distortion

On-axis aberrations of the objective are applied in its back focal plane, where position corresponds to scattering
angle θ = x/f. `PupilAberration` is a thin `GaussianActionComponent` with optical path −χ(θ):

χ(θ) = ½ C10 θ² + ½ C12 [(θx² − θy²) cos 2φ12 + 2 θx θy sin 2φ12] + ¼ C30 θ⁴.

C10 and C12 are quadratic, so their Gaussian update is exact; C30 is expanded locally about each beamlet.

Projector distortion is applied where it is simply a mapping of the image: at the image formed by `IL3`, which `PL1`
projects onto the camera. `ImageDistortion` moves every beamlet centre by r → r + ε r² r + ε_a r² ẑ×r and transforms
its slope and complex curvature canonically with the local Jacobian J (amplitude × det J^−1/2, so flux is conserved).
The illumination's converging reference wavefront (chief slope K r, read from the native matrix) is removed before
the map and restored after it, with its path, so only each beamlet's small residual phase is remapped. ε is set
directly from the fractional distortion wanted at the camera edge, and a traced ray checks it.

A phase applied near an image cannot do this with Gaussian beamlets. At the `IL3` image each beamlet is about
3 µm wide, and a few percent of distortion puts ~100 rad of non-quadratic Seidel phase across one beamlet, which
any local-quadratic update drops. The Seidel `DistortedLens` kernel fails this way here: in vacuum, 0.005 %
distortion already gave 2 % non-uniformity and 4 % gave speckle.

In [ ]:
@jdc.pytree_dataclass(kw_only=True)
class PupilAberration(GaussianActionComponent):
    z: float
    focal_length: float
    C10: float = 0.0
    C12: float = 0.0
    phi12: float = 0.0
    C30: float = 0.0

    def phase_shift(self, xy):
        tx, ty = xy[..., 0] / self.focal_length, xy[..., 1] / self.focal_length
        t2 = tx * tx + ty * ty
        astigmatism = (tx * tx - ty * ty) * jnp.cos(2 * self.phi12) + 2 * tx * ty * jnp.sin(2 * self.phi12)
        return -(0.5 * self.C10 * t2 + 0.5 * self.C12 * astigmatism + 0.25 * self.C30 * t2 * t2)

    def _call_ray(self, ray):
        xy = jnp.stack((ray.x, ray.y))
        slope = jax.grad(self.phase_shift)(xy)
        return ray.derive(dx=ray.dx + slope[0], dy=ray.dy + slope[1],
                          pathlength=ray.pathlength + self.phase_shift(xy))


f_obj = float(lenses["Obj_post"].focal_length(voltage))
pupil = PupilAberration(z=z_bfp, focal_length=f_obj, C10=20e-6, C12=5e-6, phi12=np.deg2rad(30), C30=1.2e-3)
biprism = PhaseBiprism(z=z_bip, strength=beta, width=wire, theta=image_angle,
                       sharpness=4 / 100e-9,     # 100 nm edge: below the Fresnel scale, never reaches log T = -50
                       eps=0.02)


@jdc.pytree_dataclass(kw_only=True)
class ImageDistortion(GaussianActionComponent):
    z: float
    iso: float = 0.0          # m^-2: fractional radial displacement = iso * r^2
    aniso: float = 0.0        # m^-2: fractional tangential (spiral) displacement = aniso * r^2
    reference: jnp.ndarray = jdc.field(default_factory=lambda: jnp.zeros((2, 2)))   # chief slope = K r

    def mapping(self, r):
        r2 = r @ r
        return r + self.iso * r2 * r + self.aniso * r2 * jnp.array([-r[1], r[0]])

    def _transform(self, r, d, q):
        K = jnp.asarray(self.reference)
        J = jax.jacfwd(self.mapping)(r)
        J_inv = jnp.linalg.inv(J)
        r_new = self.mapping(r)
        d_new = J_inv.T @ (d - K @ r) + K @ r_new
        q_new = J_inv.T @ (q - K) @ J_inv + K
        path = 0.5 * (r_new @ K @ r_new - r @ K @ r)
        return r_new, d_new, 0.5 * (q_new + q_new.T), jnp.linalg.det(J), path

    def _call_ray(self, ray):
        r_new, d_new, _, _, path = self._transform(
            jnp.stack((ray.x, ray.y)), jnp.stack((ray.dx, ray.dy)), jnp.zeros((2, 2)))
        return ray.derive(x=r_new[0], y=r_new[1], dx=d_new[0], dy=d_new[1], pathlength=ray.pathlength + path)

    def _call_gaussian(self, ray):
        beam = ray.to_vector()
        r_new, d_new, q_new, det, path = jax.vmap(self._transform)(beam.r_xy, beam.d_xy, beam.Q_inv)
        return beam.derive(x=r_new[:, 0], y=r_new[:, 1], dx=d_new[:, 0], dy=d_new[:, 1], Q_inv=q_new,
                           amplitude=beam.amplitude / jnp.sqrt(det), pathlength=beam.pathlength + path)


z_pl1 = float(lenses["PL1"].z)
M_pl1 = abs(np.linalg.det(matrix_to(detector.z)[:2, :2]) / np.linalg.det(matrix_to(z_il3_image)[:2, :2]))**0.5
r_edge = 0.5 * 1024 * 14e-6 / M_pl1               # IL3-image radius that lands on the camera edge
at_il3_image = matrix_to(z_il3_image)
distortion, spiral = 0.04, 0.015                  # fractional radial / tangential displacement at the camera edge
projector_distortion = ImageDistortion(
    z=z_il3_image, iso=distortion / r_edge**2, aniso=spiral / r_edge**2,
    reference=jnp.asarray(at_il3_image[2:4, :2] @ np.linalg.inv(at_il3_image[:2, :2])))

upstream = (sample, lenses["Obj_post"], pupil, Plane(z=z_bip))
downstream = (biprism, lenses["IL1"], lenses["IL2"], lenses["IL3"], projector_distortion, lenses["PL1"], detector)

theta_beamlet = wavelength / (np.pi * waist)
print(f"beamlet divergence {theta_beamlet*1e3:.2f} mrad: C10 phase {k*0.5*pupil.C10*theta_beamlet**2:.2f} rad, "
      f"C12 {k*0.5*pupil.C12*theta_beamlet**2:.2f} rad, C30 {k*0.25*pupil.C30*theta_beamlet**4:.1e} rad at that angle")
x_edge = 0.5 * 1024 * 14e-6 / M_total          # specimen point imaged onto the camera edge
probe = Ray(x=x_edge, y=0.0, dx=0.0, dy=0.0, z=z_sample, pathlength=0.0, voltage=voltage)
relay = (lenses["Obj_post"], lenses["IL1"], lenses["IL2"], lenses["IL3"])
ideal_end = run_to_end(probe, relay + (lenses["PL1"], Plane(z=detector.z)))
distorted_end = run_to_end(probe, relay + (projector_distortion, lenses["PL1"], Plane(z=detector.z)))
radial = float(np.hypot(distorted_end.x, distorted_end.y) / np.hypot(ideal_end.x, ideal_end.y) - 1)
print(f"projector distortion (traced ray): {radial:+.2%} radial at the camera edge, spiral {spiral:+.1%} "
      f"(PL1 magnification {M_pl1:.0f}; + = pincushion)")

## Optional: splitting beamlets at the wire

A specimen beamlet cannot be narrower at the wire than about the Fresnel scale √(λb'), which is also the scale of
the wire's edge fringes. Beamlets that reach the wire can optionally be split into children that are narrow across
the wire. A parent with field envelope exp(−½ rᵀPr), P = k Im Q, equals a 1D Gaussian convolution along the wire
normal e of children with covariance Σ − s² e eᵀ, s² = 1/(eᵀPe) − w_c²/2. The convolution is exact and each child
keeps the parent's local path, slope and real curvature, so only the lattice sum is discrete. Children hidden in the
opaque core are dropped. This generalises `source.decompose_gaussian` to any complex Q at any plane.

Splitting improves the wire edges but multiplies the number of Gaussians rendered on the detector, so it is off
by default; the check below measures what it buys.

In [ ]:
SPLIT_AT_WIRE = False
FIELDS = ("x", "y", "dx", "dy", "z", "pathlength", "amplitude", "Q_inv", "voltage", "_one")


def split_near_wire(beam, *, biprism, child_width=35e-9, reach=3.0, extent=4.5, overlap=1.3):
    beam = beam.to_vector()
    e = np.array([np.cos(biprism.theta), np.sin(biprism.theta)])
    k_beam = np.asarray(beam.k)
    q = np.asarray(beam.Q_inv)
    r = np.stack((np.asarray(beam.x) - biprism.x0, np.asarray(beam.y) - biprism.y0), -1)
    u = r @ e
    p_uu = k_beam * np.einsum("i,nij,j->n", e, q.imag, e)
    near = np.abs(u) < 0.5 * biprism.width + reach / np.sqrt(p_uu)
    parents, keep = np.flatnonzero(near), np.flatnonzero(~near)

    s2 = 1.0 / p_uu[parents] - 0.5 * child_width**2
    if np.any(s2 <= 0):
        raise ValueError("child_width must be narrower than every parent across the wire.")
    s = np.sqrt(s2)
    n_child = int(np.ceil(2 * extent * s.max() * overlap / child_width)) | 1   # odd: one child on centre
    unit = np.linspace(-extent, extent, n_child)
    t = (s[:, None] * unit).ravel()
    weight = ((s * (unit[1] - unit[0]))[:, None] * np.exp(-0.5 * unit**2)
              * np.sqrt((1 / s2 + 2 / child_width**2) / (2 * np.pi))[:, None]).ravel()

    covariance = np.linalg.inv(k_beam[parents, None, None] * q[parents].imag)
    covariance -= s2[:, None, None] * np.outer(e, e)
    q_child = q[parents].real + 1j * np.linalg.inv(covariance) / k_beam[parents, None, None]
    curvature_e = np.repeat(q[parents].real @ e, n_child, axis=0)          # Re Q e, per child
    e_curvature_e = curvature_e @ e

    def parent(name):
        return np.repeat(np.asarray(getattr(beam, name))[parents], n_child, axis=0)

    slope_e = parent("dx") * e[0] + parent("dy") * e[1]
    children = dict(
        x=parent("x") + t * e[0], y=parent("y") + t * e[1],
        dx=parent("dx") + curvature_e[:, 0] * t, dy=parent("dy") + curvature_e[:, 1] * t,
        z=parent("z"), voltage=parent("voltage"), _one=parent("_one"),
        pathlength=parent("pathlength") + slope_e * t + 0.5 * e_curvature_e * t**2,
        amplitude=parent("amplitude") * weight,
        Q_inv=np.repeat(q_child, n_child, axis=0),
    )
    u_child = np.repeat(u[parents], n_child) + t
    live = ((np.abs(u_child) > 0.5 * biprism.width - 3 * child_width)
            & (np.tile(np.exp(-0.5 * unit**2), parents.size) > 1e-4))
    merged = {name: jnp.concatenate((jnp.asarray(getattr(beam, name))[keep], jnp.asarray(children[name][live])))
              for name in FIELDS}
    return type(beam)(**merged, wavelength_unit=beam.wavelength_unit), dict(
        parents=parents.size, children_per_parent=n_child, children=int(live.sum()), unsplit=keep.size)

### Check: the wire against Fresnel diffraction

Without specimen and aberrations, and with a non-rotating lens of the same focal length, the field between the
wire and the first image is separable in x and y, so an exact 1D reference is available: take the Gaussian field
just above the wire, multiply it by the *same* `PhaseBiprism` transmission, and propagate it the 100 mm to the
first image with the Fresnel kernel. The y factor is fixed once from the same calculation without the biprism.

In [ ]:
def row(z, n, pitch):
    return Detector(z=z, pixel_size=(pitch, pitch), shape=(1, n))

plain_wire = replace(biprism, theta=0.0)
above_wire = propagate(beam, (Lens(z=z_obj, focal_length=f_obj), Plane(z=z_bip)))
t0 = time.perf_counter()
split_vacuum, split_counts = split_near_wire(above_wire, biprism=plain_wire)
t_split_check = time.perf_counter() - t0

n_row, pitch_row = 8192, 2e-9
render_row = lambda bundle: np.asarray(evaluate_gaussians(bundle, row(z_img, n_row, pitch_row), method="spatial"))[0]
unsplit_row = render_row(propagate(above_wire, (plain_wire, Plane(z=z_img))))
split_row = render_row(propagate(split_vacuum, (plain_wire, Plane(z=z_img))))
no_biprism_row = render_row(propagate(above_wire, (Plane(z=z_img),)))

n_ref, pitch_ref = 2**16, 1e-9
x_ref = (np.arange(n_ref) - n_ref / 2 + 0.5) * pitch_ref
field_above = np.asarray(evaluate_gaussians(above_wire, row(z_bip, n_ref, pitch_ref), method="spatial"))[0]
xy_ref = jnp.stack((jnp.asarray(x_ref), jnp.zeros(n_ref)), -1)
transmission = np.exp(1j * k * np.asarray(jax.vmap(plain_wire.phase_shift)(xy_ref))
                      + np.asarray(jax.vmap(plain_wire.log_transmission)(xy_ref)))
kernel = np.exp(-1j * np.pi * wavelength * b * np.fft.fftfreq(n_ref, pitch_ref)**2)
x_row = (np.arange(n_row) - n_row / 2 + 0.5) * pitch_row


def fresnel_to_image(field):
    out = np.fft.ifft(np.fft.fft(field) * kernel)
    return np.interp(x_row, x_ref, out.real) + 1j * np.interp(x_row, x_ref, out.imag)


reference_no_biprism = fresnel_to_image(field_above)
centre = np.abs(x_row) < 2e-6
y_factor = np.vdot(reference_no_biprism[centre], no_biprism_row[centre]) / np.vdot(
    reference_no_biprism[centre], reference_no_biprism[centre])
reference = y_factor * fresnel_to_image(field_above * transmission)

window = np.abs(x_row) < 0.5 * (2 * beta * b - shadow) + 1.5e-6
relative = lambda g: np.linalg.norm((g - reference)[window]) / np.linalg.norm(reference[window])
print(f"relative field error, specimen basis only ({n_specimen:,} Gaussians):  {relative(unsplit_row):.2%}")
print(f"relative field error, split at the wire ({split_vacuum.x.shape[0]:,} Gaussians, "
      f"split in {t_split_check:.1f} s): {relative(split_row):.2%}")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.4), constrained_layout=True, gridspec_kw=dict(width_ratios=(2, 1)))
edge_x = 0.5 * (2 * beta * b - shadow) * 1e6
for ax in axes:
    ax.plot(x_row * 1e6, abs(reference)**2, "k", lw=1.2, label="Fresnel reference")
    ax.plot(x_row * 1e6, abs(unsplit_row)**2, color="tab:blue", ls="--", lw=1, label="Gaussians, specimen basis")
    ax.plot(x_row * 1e6, abs(split_row)**2, color="tab:red", ls=":", lw=1, label="Gaussians, split at wire")
    ax.set(xlabel="first-image x (µm)", ylabel="intensity (arb.)")
axes[0].set_xlim(-edge_x - 1.5, edge_x + 1.5)
axes[1].set_xlim(edge_x - 1.2, edge_x + 1.2)
axes[0].legend(fontsize=8, loc="lower center")
axes[0].set_title("Biprism interference and wire-edge Fresnel fringes (y = 0)", loc="left")
fig.savefig(OUTPUT / "wire_check.png", dpi=150)

## Full hologram on the GPU

Each stage is timed after a warm-up call, so XLA compilation is reported separately. Complex fields are summed with
the spatially binned GPU renderer. Timings depend on anything else running on the same GPU.

In [ ]:
render = lambda bundle: evaluate_gaussians(bundle, detector, method="spatial")


def timed(fn, *args, repeats=2):
    t0 = time.perf_counter(); out = jax.block_until_ready(fn(*args)); first = time.perf_counter() - t0
    runs = []
    for _ in range(repeats):
        t0 = time.perf_counter(); out = jax.block_until_ready(fn(*args)); runs.append(time.perf_counter() - t0)
    return out, first, min(runs)


def simulate(bundle, up, down):
    above, t_up_first, t_up = timed(propagate, bundle, up)
    t0 = time.perf_counter()
    if SPLIT_AT_WIRE:
        above, _ = split_near_wire(above, biprism=down[0])
    t_split = time.perf_counter() - t0
    at_detector, t_down_first, t_down = timed(propagate, above, down)
    field, t_render_first, t_render = timed(render, at_detector)
    times = dict(up=t_up, split=t_split, down=t_down, render=t_render,
                 compile=t_up_first - t_up + t_down_first - t_down + t_render_first - t_render)
    return np.abs(np.asarray(field))**2, above.x.shape[0], times


hologram, n_rendered, times = simulate(beam, upstream, downstream)
reference_hologram, _, _ = simulate(beam, upstream[1:], downstream)      # same column, no specimen
t_total = times["up"] + times["split"] + times["down"] + times["render"]

print(device.device_kind)
print(f"  specimen Gaussians            {n_specimen:,}")
print(f"  Gaussians rendered            {n_rendered:,}  (split at wire: {SPLIT_AT_WIRE})")
print(f"  specimen → wire               {times['up']*1e3:.0f} ms")
if SPLIT_AT_WIRE:
    print(f"  split (host NumPy)            {times['split']:.2f} s")
print(f"  wire → IL1–IL3 → PL1 → camera {times['down']*1e3:.0f} ms")
print(f"  coherent render 1024²         {times['render']:.2f} s")
print(f"  total, warm                   {t_total:.2f} s   (first-call compilation {times['compile']:.1f} s)")

## The column and the simulated hologram

The column is drawn in the style of the microscope calibration pipeline's overview figures: `plot_model`'s beam
on an asinh transverse axis, so the field at the specimen and its image on the camera are both visible. It is traced from the specimen to the camera through the
microscope's own native lenses (`Obj_post`, `IL1`–`IL3`, `PL1`) in the co-rotating frame. The ray bundle spans the
recorded field (the object and reference halves) and the beamlets' angular spread. The biprism is the same
`PhaseBiprism`, given a geometric ray step from the gradient of its optical path, −β sign(u) along u. The
back-focal-plane aberrations and projector distortion are left out of this geometric drawing; at this scale they
are invisible.

In [ ]:
import dataclasses

from temgym_core.components import ElectromagneticLens

# The calibration pipeline's figure style, with text enlarged for print.
FIGURE_RC = {"font.size": 22, "axes.titlesize": 24, "axes.labelsize": 23,
             "xtick.labelsize": 20, "ytick.labelsize": 20}
ERROR_COLOURS = {"stray field": "#009E73", "distortion": "#E69F00", "on-axis": "#0072B2", "off-axis": "#6F4C9B",
                 "coherence": "#6B7280"}


@jdc.pytree_dataclass(kw_only=True)
class TracedBiprism(PhaseBiprism):
    # The same PhaseBiprism with a geometric ray step from the gradient of its optical path.
    _call_ray = PupilAberration._call_ray


diagram_biprism = TracedBiprism(**{**{f.name: getattr(biprism, f.name) for f in dataclasses.fields(biprism)},
                                   "length": 0.0})      # a numeric length for the drawing routine
diagram_column = (lenses["Obj_post"], diagram_biprism, lenses["IL1"], lenses["IL2"], lenses["IL3"],
                  lenses["PL1"], detector)
diagram_labels = ("Objective", "Biprism", "IL1", "IL2", "IL3", "PL1", "Camera")


def _length(value_m):
    for scale, unit in ((1e-3, "mm"), (1e-6, "µm"), (1e-9, "nm")):
        if value_m >= scale:
            return f"{float(f'{value_m / scale:.2g}'):g} {unit}"
    return f"{float(f'{value_m * 1e9:.2g}'):g} nm"


def draw_column(ax, *, label_fontsize=20, error_sources=False):
    # The calibration pipeline's overview look (projector_inversion._column_panel): plot_model's beam on an
    # asinh transverse axis, so the ~180 nm field at the specimen and its image on the camera are both visible.
    from matplotlib.patches import Ellipse
    from matplotlib.transforms import blended_transform_factory
    from temgym_core.plotting import legacy_beam_plot_params, plot_model, trace_ray_paths

    shifted = tuple(replace(c, z=float(c.z) - z_sample) for c in diagram_column)
    field_half = abs(cube_x) + 0.5 * 1024 * 14e-6 / M_total      # object + reference halves
    x = np.linspace(-field_half, field_half, 21)
    rays = Ray(x=x, y=0 * x, dx=0 * x, dy=0 * x, z=0.0, pathlength=0.0, voltage=voltage)
    camera_half = 0.5 * detector.shape[1] * detector.pixel_size[1]
    z_camera = float(detector.z) - z_sample
    bottom, xmax = 1.03 * z_camera, 1.06 * camera_half
    style = legacy_beam_plot_params(fixed_xmax=xmax, fill_color="#43C78E", fill_alpha=0.6,
                                    interior_ray_alpha=0.0, center_ray_alpha=0.0, ray_alpha=0.6,
                                    edge_lw=1.2, show_side_guides=False)
    plot_model(shifted, rays=rays, ax=ax, ray_coordinate="x_rot", plot_params=style)
    for artist in [*ax.texts, *ax.patches]:
        artist.remove()
    ax.set_xscale("asinh", linear_width=field_half)
    # With error labels, the optical axis sits at 62 % of the width to leave room on the left.
    centre = 0.62 if error_sources else 0.5
    reach = np.arcsinh(xmax / field_half)
    left = field_half * np.sinh(reach * centre / (1 - centre))
    ax.set(xlim=(-left, xmax), ylim=(bottom, -0.03 * bottom), xticks=[], yticks=[], xlabel="", ylabel="")
    ax.set_xticks([], minor=True)
    ax.grid(False, which="both")
    for spine in ax.spines.values():
        spine.set_visible(False)
    across = blended_transform_factory(ax.transAxes, ax.transData)
    half_width = 0.96 * (1 - centre)
    names = []
    for component, label in zip(shifted, diagram_labels):
        z = float(component.z)
        if isinstance(component, ElectromagneticLens):
            ax.add_patch(Ellipse((centre, z), 2 * half_width, 0.018 * bottom, transform=across,
                                 fill=False, lw=1.6, color="0.15", zorder=999))
        names.append((z, label))
    ax.plot([-camera_half, camera_half], [z_camera] * 2, color="0.35", lw=6, solid_capstyle="butt", zorder=999)
    z_biprism = z_bip - z_sample
    ax.plot(0, z_biprism, "o", ms=17, mfc="white", mec="black", mew=2, zorder=1002)
    ax.plot(0, 0, "o", color="#795548", ms=9, zorder=1003, clip_on=False)
    positions = [z for z, _ in names]
    for i in range(1, len(positions)):
        positions[i] = max(positions[i], positions[i - 1] + 0.05 * bottom)
    for z, (_, label) in zip(positions, names):
        ax.text(centre + half_width + 0.03, z, label, transform=across, fontsize=label_fontsize, va="center")
    if not error_sources:
        return
    style = dict(fontsize=label_fontsize, fontweight="bold", va="center", ha="right", transform=across)
    tiers = (("illumination\nspread", ERROR_COLOURS["coherence"]), ("stray field", ERROR_COLOURS["stray field"]),
             ("off-axis\naberrations", ERROR_COLOURS["off-axis"]), ("on-axis\naberrations", ERROR_COLOURS["on-axis"]))
    z_objective = z_obj - z_sample
    tier_z = [z_objective + (0.1 + 0.08 * i) * bottom for i in range(len(tiers))]
    bracket, dot = centre - half_width - 0.04, centre - half_width - 0.08
    ax.plot([centre - half_width, bracket, bracket], [z_objective, z_objective, tier_z[-1]], transform=across,
            color="0.45", lw=1.5, clip_on=False)
    for (text, colour), z in zip(tiers, tier_z):
        ax.plot([bracket, dot], [z, z], transform=across, color="0.45", lw=1.5, clip_on=False)
        ax.plot(dot, z, "o", transform=across, color=colour, ms=12, zorder=10, clip_on=False)
        ax.text(dot - 0.03, z, text, color=colour, **style)
    for text, z, colour in (("projector\ndistortion", z_il3_image - z_sample, ERROR_COLOURS["distortion"]),
                            ("shot noise,\ndetector MTF", z_camera, ERROR_COLOURS["coherence"])):
        ax.plot(centre - half_width - 0.04, z, "o", transform=across, color=colour, ms=12, zorder=10, clip_on=False)
        ax.text(centre - half_width - 0.07, z, text, color=colour, **style)


with mpl.rc_context(FIGURE_RC):
    fig = plt.figure(figsize=(18, 10), layout="constrained")
    ax_column, ax_holo = fig.subplots(1, 2, width_ratios=(0.75, 1.0))
    draw_column(ax_column)
    ax_column.set_title("(a)  Microscope column", loc="left", fontweight="bold")

    xd, yd = (np.asarray(c) * 1e3 for c in detector.coords_1d)
    extent = (xd[0], xd[-1], yd[-1], yd[0])
    image = hologram / np.median(reference_hologram)
    vmax = np.percentile(image, 99.7)
    ax_holo.imshow(image, extent=extent, cmap="gray", vmin=0, vmax=vmax)
    ax_holo.set(xlabel="Camera x (mm)", ylabel="Camera y (mm)")
    ax_holo.set_title("(b)  Simulated hologram", loc="left", fontweight="bold")
    inset = ax_holo.inset_axes((0.6, 0.6, 0.38, 0.38))
    inset.imshow(image, extent=extent, cmap="gray", vmin=0, vmax=vmax)
    inset.set(xlim=(-4.0, -1.0), ylim=(1.0, 4.0), xticks=[], yticks=[])
    for spine in inset.spines.values():
        spine.set_color("#F0E442"); spine.set_linewidth(2.5)
    ax_holo.indicate_inset_zoom(inset, edgecolor="#F0E442", lw=2)
    fig.suptitle(f"{voltage/1e3:.0f} kV  ·  {M_total:,}×  ·  {n_specimen:,} Gaussian beamlets  ·  "
                 f"{device.device_kind}: {t_total:.1f} s", x=0.01, ha="left", fontsize=21)
fig.savefig(OUTPUT / "column_and_hologram.png", dpi=200)
plt.show()

### Ideal and imperfect optics

The same specimen with ideal optics (no projector distortion, C10 = C12 = C30 = 0) beside the result above.

In [ ]:
ideal_hologram, _, _ = simulate(beam, (sample, lenses["Obj_post"], Plane(z=z_bip)),
                                tuple(c for c in downstream if c is not projector_distortion))
ideal = ideal_hologram / np.median(reference_hologram)

fig, axes = plt.subplots(1, 2, figsize=(12, 6), constrained_layout=True, sharex=True, sharey=True)
for ax, img, title in ((axes[0], ideal, "ideal optics"),
                       (axes[1], image, "PL1 distortion + C10 + C12 + C30")):
    ax.imshow(img, extent=extent, cmap="gray", vmin=0, vmax=np.percentile(img, 99.7))
    ax.set(title=title, xlabel="camera x (mm)")
axes[0].set_ylabel("camera y (mm)")
fig.savefig(OUTPUT / "ideal_vs_aberrated.png", dpi=150)

## Off-axis objective aberrations: what an isoplanatic model misses

So far the objective's aberrations are on-axis: one χ(θ) for the whole field, as in an FFT transfer-function
model. A real objective's aberrations also depend on where in the field the wave comes from, W = W(r, θ):
field curvature, field astigmatism and off-axis coma. Here the object wave and the reference wave pass the
objective at different field positions (±43 nm), and different parts of the cube see different aberrations.

The field-dependent terms are written the standard way, as a Seidel polynomial in field position o and pupil
position p. For an objective the pupil is the objective aperture in the back focal plane, p = f θ, which every
unscattered beam crosses at its centre. Referenced there, field curvature, field astigmatism and off-axis coma
leave the unscattered wave untouched and act only on what the specimen scatters:

W(o, p) = D |o|²|p|² + C (o·p)² + F |p|²(o·p) + f (o·p)(o×p).

A zero-power `SeidelLens` at `Obj_post`, with the specimen as its object plane, applies this through its joint
object/pupil action once p = (f/z₁)(a − o) is substituted into its lens-plane basis. At the objective each beamlet
occupies a small phase-space cell (3 nm × 0.27 mrad) against aberrations that vary over the whole field and
aperture, so the local expansion is well within its validity, unlike projector distortion. The coefficients are
chosen for visibility: at the object and reference positions (±43 nm) they give 2 µm of field defocus,
1.5 µm of field astigmatism, 2 µm of coma and 0.5 µm of anisotropic astigmatism.

The first figure shows the extra defocus and astigmatism each Gaussian receives across the field. The isoplanatic
model applies the same C10 and C12 everywhere.

In [ ]:
from temgym_core.aberrations import SeidelCoeffs
from temgym_core.components import SeidelLens

z1 = z_obj - z_sample


def pupil_referenced(*, field_curvature, astigmatism, coma, anisotropic_astigmatism, c):
    # W(o, p) with p = c (a - o), rewritten in SeidelLens's object/lens-plane (o, a) basis.
    D, C, F, fa = field_curvature, astigmatism, coma, anisotropic_astigmatism
    return SeidelCoeffs(A=c**2 * (D + C) - c**3 * F, B=0.0, C=c**2 * C - 2 * c**3 * F, D=c**2 * D - c**3 * F,
                        E=-2 * c**2 * (D + C) + 3 * c**3 * F, F=c**3 * F, e=-c**2 * fa, f=0.0, c=c**2 * fa)


r_ref = abs(cube_x)          # coefficients are specified at the object/reference positions
field_aberrations = SeidelLens(z=z_obj, focal_length=jnp.inf, object_plane_dist=z1, coeffs=pupil_referenced(
    field_curvature=2e-6 / (2 * f_obj**2 * r_ref**2),     # 2 µm field defocus at ±43 nm (∝ r²)
    astigmatism=1.5e-6 / (f_obj**2 * r_ref**2),           # 1.5 µm field astigmatism (∝ r²)
    coma=2e-6 / (f_obj**3 * r_ref),                       # 2 µm off-axis coma (∝ r)
    anisotropic_astigmatism=0.5e-6 / (f_obj**2 * r_ref**2),
    c=f_obj / z1))

# Local aberration received by each Gaussian: the real curvature the plate adds at the lens plane,
# expressed as specimen-side defocus and astigmatism.
at_lens = propagate(beam, (Plane(z=z_obj),))
added = np.asarray(propagate(at_lens, (field_aberrations,)).Q_inv.real - at_lens.Q_inv.real)
eig = np.linalg.eigvalsh(0.5 * (added + np.swapaxes(added, -1, -2)))
local_defocus = z1**2 * eig.mean(-1)
local_astigmatism = z1**2 * 0.5 * (eig[:, 1] - eig[:, 0])
bx, by = np.asarray(beam.x) * 1e9, np.asarray(beam.y) * 1e9
for name, x in (("object (cube) side", cube_x), ("reference side", -cube_x)):
    i = np.argmin((bx - x * 1e9)**2 + by**2)
    print(f"  {name:18s}: extra defocus {local_defocus[i]*1e6:+.2f} µm, extra astigmatism {local_astigmatism[i]*1e6:.2f} µm")

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
keep = (np.abs(bx) < 160) & (np.abs(by) < 160)
for ax, value, title in ((axes[0], local_defocus, "extra defocus per Gaussian (µm)"),
                         (axes[1], local_astigmatism, "extra astigmatism per Gaussian (µm)")):
    sc = ax.scatter(bx[keep], by[keep], c=value[keep] * 1e6, s=1, cmap="viridis", rasterized=True)
    plt.colorbar(sc, ax=ax, shrink=0.85)
    ax.add_patch(mpl.patches.Rectangle((cube_x * 1e9 - 20, -20), 40, 40, fill=False, ec="w", lw=1.5))
    for x, label in ((cube_x * 1e9, "object"), (-cube_x * 1e9, "reference")):
        ax.add_patch(mpl.patches.Circle((x, 0), 48, fill=False, ec="w", ls="--", lw=1))
        ax.text(x, 55, label, color="w", ha="center", fontsize=9)
    ax.set(title=title, xlabel="specimen x (nm)", ylabel="specimen y (nm)", aspect="equal")
fig.suptitle("Field-dependent objective aberrations seen by each Gaussian (isoplanatic model: zero everywhere)",
             fontsize=12)
fig.savefig(OUTPUT / "field_aberration_map.png", dpi=150)

### Can a reference hologram fix off-axis aberrations?

Three holograms of the cube, all through the same column including projector distortion, each with a vacuum
reference hologram recorded through the same optics:

* a perfect objective (the target),
* the objective with **off-axis aberrations only**.

Each hologram is reconstructed in the standard way: isolate the sideband, re-centre it, transform back, and divide
by the same reconstruction of the vacuum reference hologram.

In [ ]:
def sideband(hologram, reference, radius_fraction=0.33):
    # Sideband reconstruction; returns the specimen and vacuum waves before division.
    spectrum, reference_spectrum = np.fft.fft2(hologram), np.fft.fft2(reference)
    n_y, n_x = hologram.shape
    ky, kx = np.meshgrid(np.fft.fftfreq(n_y) * n_y, np.fft.fftfreq(n_x) * n_x, indexing="ij")
    carrier = np.unravel_index(np.argmax(np.abs(reference_spectrum) * (kx > 20)), hologram.shape)
    cy, cx = ky[carrier], kx[carrier]
    mask = (kx - cx)**2 + (ky - cy)**2 < (radius_fraction * np.hypot(cx, cy))**2
    centred = lambda s: np.fft.ifft2(np.roll(s * mask, (-int(cy), -int(cx)), axis=(0, 1)))
    return centred(spectrum), centred(reference_spectrum)


perfect_objective = (sample, lenses["Obj_post"], Plane(z=z_bip))
columns = {
    "no aberrations": perfect_objective,
    "off-axis aberrations": (sample, field_aberrations, lenses["Obj_post"], Plane(z=z_bip)),
}
raw, corrected, recorded = {}, {}, {}
for name, column in columns.items():
    specimen_hologram, _, _ = simulate(beam, column, downstream)
    vacuum_hologram, _, _ = simulate(beam, column[1:], downstream)
    specimen_wave, vacuum_wave = sideband(specimen_hologram, vacuum_hologram)
    recorded[name] = (specimen_hologram, vacuum_hologram)
    raw[name], corrected[name] = specimen_wave, specimen_wave / vacuum_wave
    if name == "no aberrations":
        support = np.abs(vacuum_wave) > 0.3 * np.abs(vacuum_wave).max()
        raw_target = specimen_wave
target = corrected["no aberrations"]

cam_x, cam_y = (np.asarray(c) * 1e3 for c in detector.coords_1d)
CX, CY = np.meshgrid(cam_x, cam_y)
cube_region = support & (np.abs(CX) < 4) & (np.abs(CY) < 3.5)


def relative_error(wave, goal=target, region=cube_region):
    # Relative field error after removing a free global phase and amplitude.
    scale = np.vdot(wave[region], goal[region]) / np.vdot(wave[region], wave[region])
    return float(np.linalg.norm((scale * wave - goal)[region]) / np.linalg.norm(goal[region]))


name = "off-axis aberrations"
print(f"relative field error in the cube region, against the aberration-free reconstruction:\n"
      f"  raw {relative_error(raw[name], raw_target):.3f}, after the reference hologram {relative_error(corrected[name]):.3f}")

remaining = np.angle(corrected["off-axis aberrations"] / target)
cube_view = dict(xlim=(-4.5, 4.5), ylim=(-4.5, 4.5))
fig, axes = plt.subplots(1, 4, figsize=(22, 5.6), constrained_layout=True)
panels = (
    (np.angle(target), "no aberrations\n(reference-corrected)"),
    (np.angle(raw["off-axis aberrations"]), "off-axis aberrations, raw\n(no reference hologram)"),
    (np.angle(corrected["off-axis aberrations"]), "off-axis aberrations\n(reference-corrected)"),
)
for ax, (data, title) in zip(axes[:3], panels):
    ax.imshow(np.where(support, data, np.nan), extent=extent, cmap="twilight", vmin=-np.pi, vmax=np.pi)
    ax.set(title=title, **cube_view)
im = axes[3].imshow(np.where(support, remaining, np.nan), extent=extent, cmap="RdBu_r", vmin=-1, vmax=1)
axes[3].set(title=f"left after the reference hologram\n(off-axis − no aberrations) "
                  f"{np.sqrt(np.mean(remaining[cube_region]**2)):.2f} rad rms", **cube_view)
plt.colorbar(im, ax=axes[3], shrink=0.85, label="phase (rad)")
for ax in axes:
    ax.set(xlabel="camera x (mm)", ylabel="camera y (mm)")
fig.suptitle("Reconstructed phase of the magnetic cube: the reference hologram does not remove off-axis aberrations",
             fontsize=13, y=1.07)
fig.savefig(OUTPUT / "offaxis_reference_hologram.png", dpi=150, bbox_inches="tight")

The two recorded holograms behind the off-axis reconstruction: the specimen hologram through the aberrated objective,
and the vacuum reference hologram recorded through exactly the same optics with the specimen removed.

In [ ]:
specimen_hologram, vacuum_hologram = recorded["off-axis aberrations"]
scale = np.median(vacuum_hologram)
zoom = (-3.6, -1.6, -1.0, 1.0)            # mm: across the cube's left edge, where the aberrations are largest
fig, axes = plt.subplots(1, 2, figsize=(15, 7.4), constrained_layout=True)
for ax, data, title in ((axes[0], specimen_hologram, "specimen hologram, off-axis aberrations"),
                        (axes[1], vacuum_hologram, "vacuum reference hologram, same optics")):
    image = data / scale
    vmax = np.percentile(specimen_hologram / scale, 99.7)
    ax.imshow(image, extent=extent, cmap="gray", vmin=0, vmax=vmax)
    ax.set(title=title, xlabel="camera x (mm)", ylabel="camera y (mm)")
    inset = ax.inset_axes((0.6, 0.6, 0.38, 0.38))
    inset.imshow(image, extent=extent, cmap="gray", vmin=0, vmax=vmax)
    inset.set(xlim=zoom[:2], ylim=zoom[2:], xticks=[], yticks=[])
    for spine in inset.spines.values():
        spine.set_color("#F0E442"); spine.set_linewidth(2)
    ax.indicate_inset_zoom(inset, edgecolor="#F0E442", lw=1.5)
fig.suptitle("Recorded holograms (off-axis objective aberrations, projector distortion included)", fontsize=13)
fig.savefig(OUTPUT / "offaxis_holograms.png", dpi=150, bbox_inches="tight")

### Does a reference hologram remove these aberrations?

Only partly. A vacuum reference hologram recorded through the same optics divides out what acts identically on
both exposures, which means what happens to the *unscattered* waves:

| Effect | Removed by the reference hologram? |
|---|---|
| Fringe bending from projector distortion, biprism irregularities, the lenses' phase on the vacuum wave | yes |
| Smooth field phase that off-axis aberrations impose on the unscattered wave | yes (here it largely cancels anyway, because object and reference sit symmetrically about the axis) |
| Geometric distortion of the image itself | no: the phase is right but at displaced positions, so it needs a geometric correction |
| On-axis C10/C12/C30 acting on the specimen's scattered wave | no, but a known χ(θ) can be deconvolved because it is the same everywhere |
| Field-dependent defocus, astigmatism and coma acting on the scattered wave | no, and a single χ(θ) cannot deconvolve it either, because it changes across the field |

The last row is what remains in the figure above: it sits at the cube's edges, where the specimen scatters, and the
reference hologram cannot remove it. Nor can a single transfer function χ(θ) deconvolve it exactly, because the
aberration changes with position across the image; a position-dependent correction needs a model like this one. Correcting it needs a space-variant model of the objective. The Gaussian beamlets
provide one without extra cost, because each beamlet already receives its own local aberration.

## The cube's stray field and the perturbed reference wave

The cube above has its vector potential confined to its volume, so it has no external field. A real uniformly
magnetised particle returns its flux outside itself, and the projected phase of that return field falls off only
as ~1/r in the image plane. The "vacuum" reference region, about 87 nm from the cube here, therefore carries some
of the cube's own phase, and dividing by the reference wave subtracts it from the measurement.

Here the specimen is a 50 nm **magnetite** cube, the classic single-domain case: µ₀Ms = 0.6 T and a 17 V mean inner
potential, tilted by 20° about x and magnetised along its own y axis. Its magnetic phase, including the external
field, follows from the projected shape in Fourier space (Beleggia & Zhu):

φ_m(k) = i (e/ħ) µ₀Ms (m × k)_z D(k) / k²,

where D(k) is the transform of the projected thickness and k the angular spatial frequency. The projected induction
obeys two-dimensional magnetostatics, so a uniformly magnetised disc (or infinite cylinder) has exactly half the
naive interior phase gradient, (e/ħ)µ₀Ms t / 2. The first check below confirms that.

In [ ]:
from scipy.interpolate import RegularGridInterpolator
from temgym_core.constants import e_over_hbar, interaction_constant_CE

E_OVER_HBAR = float(e_over_hbar())


def magnetic_phase(thickness, pitch, m_xy, B0):
    # Projected magnetic phase of a uniformly magnetised body, external (stray) field included.
    n_y, n_x = thickness.shape
    ky, kx = np.meshgrid(2 * np.pi * np.fft.fftfreq(n_y, pitch), 2 * np.pi * np.fft.fftfreq(n_x, pitch),
                         indexing="ij")
    k2 = kx**2 + ky**2
    k2[0, 0] = 1.0
    shape = np.fft.fft2(thickness)
    phase_k = 1j * E_OVER_HBAR * B0 * shape * (m_xy[0] * ky - m_xy[1] * kx) / k2
    phase_k[0, 0] = 0.0
    return np.real(np.fft.ifft2(phase_k))


n_map, pitch_map = 2048, 1e-9                     # 2 µm field so the 1/r tail is represented
axis = (np.arange(n_map) - n_map / 2) * pitch_map
X, Y = np.meshgrid(axis, axis)

disc_t = 10e-9
disc_phase = magnetic_phase(disc_t * (X**2 + Y**2 < (60e-9)**2), pitch_map, (0.0, 1.0), 1.0)
disc_gradient = np.gradient(disc_phase, pitch_map, axis=1)[n_map // 2, n_map // 2]
print(f"check, uniformly magnetised disc: interior gradient / ((e/ħ) B0 t) = "
      f"{disc_gradient / (E_OVER_HBAR * disc_t):.4f} (2-D magnetostatics: exactly 1/2)")

# Projected thickness of the tilted 50 nm cube, centred at cube_x.
side, cube_tilt = 50e-9, np.deg2rad(20)
z_line = np.linspace(-side, side, 4001)
cos_t, sin_t = np.cos(cube_tilt), np.sin(cube_tilt)
y_obj = cos_t * axis[:, None] + sin_t * z_line[None, :]
z_obj_frame = -sin_t * axis[:, None] + cos_t * z_line[None, :]
chord = np.sum((np.abs(y_obj) <= side / 2) & (np.abs(z_obj_frame) <= side / 2), axis=1) * (z_line[1] - z_line[0])
projected_thickness = chord[:, None] * (np.abs(axis[None, :] - cube_x) <= side / 2)

mu0_Ms, mean_inner_potential = 0.6, 17.0          # magnetite
magnetisation = (0.0, cos_t)                        # in-plane part of the cube's y axis after the tilt
phi_magnetic = gaussian_filter(magnetic_phase(projected_thickness, pitch_map, magnetisation, mu0_Ms), 1.5)
phi_electric = gaussian_filter(float(interaction_constant_CE(voltage)) * mean_inner_potential
                               * projected_thickness, 1.5)
log_amplitude_stray = gaussian_filter(np.log(0.8) * projected_thickness / side, 1.5)
phi_total = phi_electric + phi_magnetic
print(f"magnetic phase range {np.ptp(phi_magnetic):.2f} rad (naive (e/ħ)µ0Ms·t·w = "
      f"{E_OVER_HBAR * mu0_Ms * side * side:.2f} rad), electrostatic up to {phi_electric.max():.2f} rad")

separation = 2 * abs(cube_x)                        # object-to-reference distance at the specimen
view = np.abs(axis) <= 250e-9
fig, axes = plt.subplots(1, 2, figsize=(14, 5.4), constrained_layout=True,
                         gridspec_kw=dict(width_ratios=(1.0, 1.35)))
ext_map = np.array([axis[view][0], axis[view][-1], axis[view][0], axis[view][-1]]) * 1e9
axes[0].imshow(np.cos(12 * phi_magnetic[np.ix_(view, view)]), extent=ext_map, origin="lower", cmap="gray")
for x, label in ((cube_x, "object"), (cube_x + separation, "reference")):
    axes[0].add_patch(mpl.patches.Circle((x * 1e9, 0), 48, fill=False, ec="#F0E442", ls="--", lw=1.2))
    axes[0].text(x * 1e9, 56, label, color="#F0E442", ha="center", fontsize=9)
axes[0].set(title="magnetic phase contours, cos(12 φ_m): flux lines incl. stray field",
            xlabel="specimen x (nm)", ylabel="specimen y (nm)")
row = n_map // 2
axes[1].plot(axis[view] * 1e9, phi_magnetic[row, view], color="#0072B2", lw=2, label="magnetic phase φ_m")
axes[1].axvspan((cube_x - side / 2) * 1e9, (cube_x + side / 2) * 1e9, color="#CC79A7", alpha=0.15, label="cube")
axes[1].axvspan((cube_x + separation - 25e-9) * 1e9, (cube_x + separation + 25e-9) * 1e9, color="#F0E442",
                alpha=0.3, label="reference for the cube (87 nm away)")
axes[1].axhline(0, color="#9CA3AF", lw=0.8)
axes[1].set(xlabel="specimen x (nm), y = 0", ylabel="phase (rad)",
            title="the stray field's phase extends well outside the cube")
axes[1].legend(frameon=False, fontsize=9)
fig.savefig(OUTPUT / "stray_field_phase.png", dpi=150)

### How much of the cube's phase does the reference wave carry?

Off-axis holography measures φ(r) − φ(r + d), with d the object-to-reference displacement at the specimen. Over the
cube, the error is therefore minus the stray phase at the displaced reference positions. The curve below takes it
straight from the phase map for different displacements (perpendicular to m, the worst case), as recorded and after
the common step of fitting and removing a linear ramp.

In [ ]:
phase_at = RegularGridInterpolator((axis, axis), phi_magnetic, bounds_error=False, fill_value=0.0)
rows_c, cols_c = np.nonzero(projected_thickness > 0)
cube_points = np.stack((axis[rows_c], axis[cols_c]), -1)          # (y, x)
design = np.c_[np.ones(len(cube_points)), cube_points]
distances = np.geomspace(60e-9, 800e-9, 40)       # displaced points stay inside the 2 µm map
raw_span, ramp_span = [], []
for d in distances:
    stray = phase_at(cube_points + np.array([0.0, d]))
    residual = stray - design @ np.linalg.lstsq(design, stray, rcond=None)[0]
    raw_span.append(np.ptp(stray)); ramp_span.append(np.ptp(residual))
raw_span, ramp_span = np.array(raw_span), np.array(ramp_span)
signal = np.ptp(phi_magnetic[projected_thickness > 0])
at_ours = np.interp(separation, distances, raw_span)
print(f"at our {separation*1e9:.0f} nm separation: {at_ours:.3f} rad across the cube "
      f"({at_ours/signal:.0%} of its {signal:.2f} rad magnetic signal), "
      f"{np.interp(separation, distances, ramp_span):.3f} rad after ramp removal")

fig, ax = plt.subplots(figsize=(8, 4.6), constrained_layout=True)
ax.loglog(distances * 1e9, raw_span / signal, color="#0072B2", lw=2, label="as recorded")
ax.loglog(distances * 1e9, ramp_span / signal, color="#D55E00", lw=2, label="after removing a fitted ramp")
ax.axvline(separation * 1e9, color="#6B7280", ls="--", lw=1)
ax.text(separation * 1e9 * 1.05, 2e-4, "this hologram", color="#6B7280", fontsize=9)
ax.set(xlabel="object-to-reference distance (nm)", ylabel="stray phase across the cube / magnetic signal",
       title="Reference-wave perturbation from a 50 nm magnetite cube")
ax.grid(True, which="both", color="#E5E7EB", lw=0.5)
ax.legend(frameon=False)
fig.savefig(OUTPUT / "stray_field_vs_distance.png", dpi=150)

### The same effect in the simulated hologram

Two holograms through the column with ideal optics (to isolate this effect): **A** with the cube's full field,
and **B** identical except that the specimen phase on the reference side (x > 0) is removed, i.e. a perfectly
clean reference. Both are reconstructed with the same vacuum reference hologram. Everything else, including
reconstruction resolution and the Gaussian approximation, cancels in A − B, which should equal the predicted
−φ_m at the reference positions.

In [ ]:
keep = np.abs(axis) <= 400e-9
def specimen_from(phase):
    return InterpolatedSample2D.from_action_maps(
        pathlength_yx_m=phase[np.ix_(keep, keep)] / k, log_amplitude_yx=log_amplitude_stray[np.ix_(keep, keep)],
        x_coords_m=axis[keep], y_coords_m=axis[keep], z=z_sample)

clean_reference_side = 0.5 * (1 - np.tanh(X / 3e-9))            # 1 on the object side, 0 on the reference side
ideal_projection = tuple(c for c in downstream if c is not projector_distortion)
waves = {}
vacuum_hologram, _, _ = simulate(beam, (lenses["Obj_post"], Plane(z=z_bip)), ideal_projection)
for name, phase in (("A: full stray field", phi_total), ("B: clean reference", phi_total * clean_reference_side)):
    hologram_s, _, _ = simulate(beam, (specimen_from(phase), lenses["Obj_post"], Plane(z=z_bip)), ideal_projection)
    specimen_wave, vacuum_wave = sideband(hologram_s, vacuum_hologram)
    waves[name] = specimen_wave / vacuum_wave

measured = np.angle(waves["A: full stray field"] / waves["B: clean reference"])
cam_x, cam_y = (np.asarray(c) for c in detector.coords_1d)
CXm, CYm = np.meshgrid(cam_x, cam_y)
object_x, object_y = CXm / M_total + cube_x, CYm / M_total           # specimen position of the object wave
predicted = -phase_at(np.stack((object_y, object_x + separation), -1))
support_s = np.abs(vacuum_wave) > 0.3 * np.abs(vacuum_wave).max()
compare = support_s & (object_x < -10e-9)                             # away from the object/reference midline
offset = np.angle(np.mean(np.exp(1j * (measured - predicted))[compare]))
measured = measured - offset
print(f"A − B vs prediction over the object region: rms difference "
      f"{np.sqrt(np.mean((measured - predicted)[compare]**2)):.4f} rad, signal rms {np.std(predicted[compare]):.4f} rad")

fig, axes = plt.subplots(1, 3, figsize=(19, 5.2), constrained_layout=True)
lim = np.abs(predicted[compare]).max()
for ax, data, title in ((axes[0], measured, "simulated: A − B (reconstructed phase)"),
                        (axes[1], predicted, "predicted: −φ_m at the reference positions")):
    im = ax.imshow(np.where(compare, data, np.nan), extent=extent, cmap="RdBu_r", vmin=-lim, vmax=lim)
    ax.set(title=title, xlabel="camera x (mm)", ylabel="camera y (mm)")
plt.colorbar(im, ax=axes[:2], shrink=0.85, label="phase (rad)")
row = np.argmin(np.abs(cam_y))
axes[2].plot(object_x[row] * 1e9, predicted[row], color="k", lw=2, label="predicted")
axes[2].plot(object_x[row] * 1e9, np.where(compare[row], measured[row], np.nan), color="#D55E00", ls="--", lw=1.5,
             label="simulated A − B")
axes[2].axvspan((cube_x - side / 2) * 1e9, (cube_x + side / 2) * 1e9, color="#CC79A7", alpha=0.15, label="cube")
axes[2].set(xlabel="specimen x of the object wave (nm), y = 0", ylabel="phase error (rad)",
            title="phase error the stray field puts in the reconstruction")
axes[2].legend(frameon=False, fontsize=9)
fig.savefig(OUTPUT / "stray_field_reference_error.png", dpi=150)

### What is a reasonable stray field?

For a single-domain magnetite cube of 50 nm (µ₀Ms = 0.6 T), the field just outside a face is of order µ₀Ms/2
(~0.3 T), falling as ~(a/r)³ in three dimensions. What the hologram sees is the projected phase, which falls as
~a/r. At this 87 nm object-to-reference distance the reference carries about 12 % of the cube's magnetic phase
range across the cube. It is smooth, so fitting and removing a ramp roughly halves it (to ~5 %); beyond about
200 nm the remainder is a fraction of a percent, and by several hundred nanometres the raw error is below 1 %.
A removed ramp also removes any genuine ramp in the specimen phase, so this correction is not free. Materials with larger magnetisation scale the effect proportionally while they stay single-domain;
larger particles of iron, cobalt or permalloy tend to form vortices, which close their flux internally and have
little stray field.

## Summary: realistic optics, holograms and measurement

One figure with **realistic, small** imperfections, using the single-domain magnetite cube with its full stray field:

| Effect | Value used |
|---|---|
| On-axis objective aberrations | C10 = 200 nm, C12 = 20 nm, C30 = 1.2 mm |
| Off-axis objective aberrations | order-unity coefficients in units of f: field defocus ≈ r²/f, field astigmatism ≈ r²/f, coma ≈ r (estimates, not measured) |
| Projector distortion | 1 % radial, 0.3 % spiral at the camera edge |
| Biprism | ±35 µrad, 0.6 µm wire, 87 nm object-to-reference distance |
| Recording | illumination spread 7 µrad across the biprism (partial spatial coherence, 7-point incoherent sum), detector blur 0.7 px, Poisson noise at 300 e⁻/pixel |

The magnetic phase is measured as in experiments, from the realistic recordings: two holograms with the
magnetisation reversed, each divided by the same recorded vacuum reference hologram; half the phase difference is
the magnetic phase. The optics terms of the error budget are computed from noise-free coherent holograms so each is
isolated; the recording term is the change from the coherent to the recorded measurement. The error budget switches each
effect on alone and reports the rms change it makes to the measured magnetic phase over the cube (a differential
measure, so the simulation's own floor cancels); the floor itself is reported against the true phase.

In [ ]:
t_summary = time.perf_counter()
realistic_pupil = replace(pupil, C10=200e-9, C12=20e-9, C30=1.2e-3)
realistic_off_axis = SeidelLens(z=z_obj, focal_length=jnp.inf, object_plane_dist=z1, coeffs=pupil_referenced(
    field_curvature=1 / (2 * f_obj**3), astigmatism=1 / f_obj**3, coma=1 / f_obj**3,
    anisotropic_astigmatism=0.5 / f_obj**3, c=f_obj / z1))
realistic_distortion = replace(projector_distortion, iso=0.01 / r_edge**2, aniso=0.003 / r_edge**2)

effects = ("on-axis", "off-axis", "stray field", "distortion")
configurations = {"ideal": set(), **{name: {name} for name in effects}, "all": set(effects)}


def column_for(active, specimen):
    upstream_ = ((specimen,) if specimen is not None else ()) + (
        (realistic_off_axis,) if "off-axis" in active else ()) + (lenses["Obj_post"],) + (
        (realistic_pupil,) if "on-axis" in active else ()) + (Plane(z=z_bip),)
    downstream_ = (biprism, lenses["IL1"], lenses["IL2"], lenses["IL3"]) + (
        (realistic_distortion,) if "distortion" in active else ()) + (lenses["PL1"], detector)
    return upstream_ + downstream_


def magnetite(sign, active):
    phase = phi_electric + sign * phi_magnetic
    if "stray field" not in active:
        phase = phase * clean_reference_side
    return specimen_from(phase)


magnetic_measured, recorded_summary, vacuum_cache = {}, {}, {}
for name, active in configurations.items():
    optics_key = tuple(sorted(active - {"stray field"}))
    if optics_key not in vacuum_cache:
        vacuum_cache[optics_key] = np.abs(np.asarray(render(propagate(beam, column_for(active, None)))))**2
    vacuum_holo = vacuum_cache[optics_key]
    waves_pm = []
    for sign in (+1, -1):
        holo = np.abs(np.asarray(render(propagate(beam, column_for(active, magnetite(sign, active))))))**2
        specimen_wave, vacuum_wave = sideband(holo, vacuum_holo)
        waves_pm.append(specimen_wave / vacuum_wave)
        if name == "all" and sign == +1:
            recorded_summary = dict(specimen=holo, vacuum=vacuum_holo)
    magnetic_measured[name] = 0.5 * np.angle(waves_pm[0] / waves_pm[1])
support_summary = np.abs(vacuum_wave) > 0.3 * np.abs(vacuum_wave).max()


# A realistic recording of the all-effects hologram: partial spatial coherence (an incoherent sum over
# illumination tilts across the biprism, Gauss–Hermite quadrature), detector blur and shot noise.
illumination_spread = 7e-6        # rad, 1 sigma across the biprism
dose = 300.0                      # electrons per pixel in the vacuum reference
mtf_sigma = 0.7                   # detector blur, pixels
noise = np.random.default_rng(2026)
tilt_nodes, tilt_weights = np.polynomial.hermite_e.hermegauss(7)
tilt_weights = tilt_weights / tilt_weights.sum()


def partially_coherent(active, specimen):
    column = column_for(active, specimen)
    total = 0.0
    for node, weight in zip(tilt_nodes, tilt_weights):
        tilt = illumination_spread * node
        tilted = beam.derive(dx=beam.dx + tilt, pathlength=beam.pathlength + tilt * beam.x)
        total = total + weight * np.abs(np.asarray(render(propagate(tilted, column))))**2
    return total


def detect(intensity, electrons_per_unit):
    return noise.poisson(np.clip(gaussian_filter(intensity, mtf_sigma) * electrons_per_unit, 0, None)).astype(float)


vacuum_expected = partially_coherent(set(effects), None)
electrons = dose / np.median(vacuum_expected)
vacuum_recorded = detect(vacuum_expected, electrons)
recorded_pm = [detect(partially_coherent(set(effects), magnetite(sign, set(effects))), electrons)
               for sign in (+1, -1)]
waves_recorded = [np.divide(*sideband(h, vacuum_recorded)) for h in recorded_pm]
magnetic_measured["recorded"] = 0.5 * np.angle(waves_recorded[0] / waves_recorded[1])
recorded_summary = dict(specimen=recorded_pm[0], vacuum=vacuum_recorded)
spectrum_v = np.abs(np.fft.fft2(vacuum_expected))
kx_v = np.fft.fftfreq(spectrum_v.shape[1]) * spectrum_v.shape[1]
carrier_v = np.unravel_index(np.argmax(spectrum_v * (kx_v[None, :] > 20)), spectrum_v.shape)
visibility = 2 * spectrum_v[carrier_v] / spectrum_v[0, 0]


def band_limited(image, reference=vacuum_cache[()], radius_fraction=0.33):
    # Apply the same circular band limit as the sideband reconstruction.
    spectrum = np.fft.fft2(reference)
    n_y, n_x = image.shape
    ky, kx = np.meshgrid(np.fft.fftfreq(n_y) * n_y, np.fft.fftfreq(n_x) * n_x, indexing="ij")
    carrier = np.unravel_index(np.argmax(np.abs(spectrum) * (kx > 20)), image.shape)
    radius = radius_fraction * np.hypot(ky[carrier], kx[carrier])
    return np.real(np.fft.ifft2(np.fft.fft2(image) * (kx**2 + ky**2 < radius**2)))


cam_x, cam_y = (np.asarray(c) for c in detector.coords_1d)
CXs, CYs = np.meshgrid(cam_x, cam_y)
obj_x, obj_y = CXs / M_total + cube_x, CYs / M_total
thickness_at = RegularGridInterpolator((axis, axis), projected_thickness, bounds_error=False, fill_value=0.0)
in_cube = support_summary & (thickness_at(np.stack((obj_y, obj_x), -1)) > 0.5 * side)
true_magnetic = band_limited(phase_at(np.stack((obj_y, obj_x), -1)))
centre = lambda a: a - np.mean(a[in_cube])
rms_in_cube = lambda a: float(np.sqrt(np.mean(centre(a)[in_cube]**2)))
signal_m = np.ptp(true_magnetic[in_cube])
floor = rms_in_cube(magnetic_measured["ideal"] - true_magnetic)
budget = {name: rms_in_cube(magnetic_measured[name] - magnetic_measured["ideal"]) for name in effects}
budget["coherence"] = rms_in_cube(magnetic_measured["recorded"] - magnetic_measured["all"])
budget["all"] = rms_in_cube(magnetic_measured["recorded"] - magnetic_measured["ideal"])
stray_at_250 = np.interp(250e-9, distances, raw_span)
print(f"recording: illumination spread {illumination_spread*1e6:.0f} µrad, fringe visibility {visibility:.2f}, "
      f"{dose:.0f} e⁻/pixel, detector blur {mtf_sigma} px")
print(f"{device.device_kind}: {len(configurations) * 2 + len(vacuum_cache) + 3 * len(tilt_nodes)} holograms "
      f"of {n_specimen:,} Gaussians in {time.perf_counter() - t_summary:.0f} s")
print(f"magnetic signal across the cube {signal_m:.2f} rad; simulation + reconstruction floor vs truth {floor:.4f} rad rms")
for name, value in budget.items():
    print(f"  {name:12s}: {value:.2e} rad rms ({value / signal_m:.2%} of the signal)")
print(f"stray-field error across the cube at a more typical 250 nm distance: {stray_at_250:.3f} rad "
      f"({stray_at_250 / signal:.1%})")

### Sensitivity budget by differentiation

Every step from the column parameters to the measured magnetic phase is differentiable in JAX: the Gaussian
propagation, the coherent sum on the camera (the `scan` renderer) and the sideband reconstruction. The magnetic
phase ½ arg(S₊ S₋*) of the magnetisation-reversed pair does not need the vacuum hologram. So a single forward-mode
`jax.jacfwd` gives the derivative of the measured magnetic phase at every camera pixel with respect to seven
controls at once: the five lens currents, the biprism deflection and an illumination tilt. It is evaluated on the
central 512² of the camera. The rms change over the cube, set against 1 % of the magnetic signal, gives the
tolerance on each control. The derivatives are local; they describe small drifts about this operating point.
This section is optional and off by default (`RUN_SENSITIVITY`); its tolerances are not yet compared with the
measured stability of real lens and biprism supplies, so they are not part of the summary figure.

In [ ]:
RUN_SENSITIVITY = False      # optional: about 4 minutes on the GPU
if RUN_SENSITIVITY:
    t_sensitivity = time.perf_counter()
    crop = replace(detector, shape=(512, 512))
    lens_names = ("Obj_post", "IL1", "IL2", "IL3", "PL1")
    control_names = (*lens_names, "biprism", "tilt")
    all_effects = set(effects)
    magnetite_pm = {sign: magnetite(sign, all_effects) for sign in (+1, -1)}


    def column_with(p):
        # p: relative changes of the five lens currents and the biprism deflection; p[6] is a tilt in rad.
        adjusted = []
        for component in column_for(all_effects, None):
            for i, name in enumerate(lens_names):
                if component is lenses[name]:
                    component = replace(component, current=component.current * (1 + p[i]))
            if component is biprism:
                component = replace(component, strength=component.strength * (1 + p[5]))
            adjusted.append(component)
        return tuple(adjusted)


    def differentiable_hologram(p, sign):
        tilted = beam.derive(dx=beam.dx + p[6], pathlength=beam.pathlength + p[6] * beam.x)
        at_camera = run_to_end(tilted, (magnetite_pm[sign],) + column_with(p))
        return jnp.abs(evaluate_gaussians(at_camera, crop, method="scan", batch_size=32))**2


    nominal = np.asarray(jax.jit(lambda p: differentiable_hologram(p, +1))(jnp.zeros(7)))
    n_crop = nominal.shape[0]
    ky_c, kx_c = np.meshgrid(np.fft.fftfreq(n_crop) * n_crop, np.fft.fftfreq(n_crop) * n_crop, indexing="ij")
    carrier_c = np.unravel_index(np.argmax(np.abs(np.fft.fft2(nominal)) * (kx_c > 10)), nominal.shape)
    cy_c, cx_c = int(ky_c[carrier_c]), int(kx_c[carrier_c])
    sideband_mask = jnp.asarray((kx_c - cx_c)**2 + (ky_c - cy_c)**2 < (0.33 * np.hypot(cx_c, cy_c))**2)


    def measured_magnetic(p):
        sideband_of = lambda h: jnp.fft.ifft2(jnp.roll(jnp.fft.fft2(h) * sideband_mask, (-cy_c, -cx_c), axis=(0, 1)))
        return 0.5 * jnp.angle(sideband_of(differentiable_hologram(p, +1))
                               * jnp.conj(sideband_of(differentiable_hologram(p, -1))))


    jacobian = np.asarray(jax.jit(jax.jacfwd(measured_magnetic))(jnp.zeros(7)))    # (512, 512, 7)
    crop_x, crop_y = (np.asarray(c) for c in crop.coords_1d)
    crop_obj_x, crop_obj_y = np.meshgrid(crop_x / M_total + cube_x, crop_y / M_total)
    crop_cube = thickness_at(np.stack((crop_obj_y, crop_obj_x), -1)) > 0.5 * side
    per_unit = np.array([np.std(jacobian[..., i][crop_cube]) for i in range(7)])   # rad per unit change
    tolerance = 0.01 * signal_m / per_unit                                             # change giving a 1 % error
    print(f"jacfwd of the measured magnetic phase w.r.t. 7 controls: {time.perf_counter() - t_sensitivity:.0f} s")
    for name, value in zip(control_names, tolerance):
        shown = f"{value * 1e6:.2f} µrad" if name == "tilt" else f"{value * 1e6:.0f} ppm"
        print(f"  {name:8s}: a 1 % error in the magnetic phase needs a change of {shown}")

    # Check one derivative against a central finite difference through the same pipeline.
    step = np.zeros(7); step[2] = 1e-4
    phase_at_p = jax.jit(measured_magnetic)
    finite = (np.asarray(phase_at_p(jnp.asarray(step))) - np.asarray(phase_at_p(jnp.asarray(-step)))) / 2e-4
    print(f"check (IL2): jacfwd vs finite difference, relative difference "
          f"{np.linalg.norm((jacobian[..., 2] - finite)[crop_cube]) / np.linalg.norm(finite[crop_cube]):.1%}")

In [ ]:
with mpl.rc_context(FIGURE_RC):
    fig = plt.figure(figsize=(32, 18), layout="constrained")
    fig.get_layout_engine().set(w_pad=0.25, h_pad=0.3, wspace=0.06, hspace=0.08)
    grid = fig.add_gridspec(2, 4, width_ratios=(1.15, 1.0, 1.0, 1.0))
    ax_column = fig.add_subplot(grid[:, 0])
    ax_holo, ax_truth, ax_map = (fig.add_subplot(grid[0, i]) for i in (1, 2, 3))
    lower = grid[1, 1:].subgridspec(1, 2, width_ratios=(1.0, 1.0), wspace=0.12)
    ax_profile, ax_budget = fig.add_subplot(lower[0]), fig.add_subplot(lower[1])
    draw_column(ax_column, error_sources=True)

    xd, yd = cam_x * 1e3, cam_y * 1e3
    ext = (xd[0], xd[-1], yd[-1], yd[0])
    scale_v = np.median(recorded_summary["vacuum"])
    image = recorded_summary["specimen"] / scale_v
    vmax = np.percentile(image, 99.7)
    ax_holo.imshow(image, extent=ext, cmap="gray", vmin=0, vmax=vmax)
    ax_holo.set(xlabel="Camera x (mm)", ylabel="Camera y (mm)")
    inset = ax_holo.inset_axes((0.58, 0.58, 0.4, 0.4))
    inset.imshow(image, extent=ext, cmap="gray", vmin=0, vmax=vmax)
    inset.set(xlim=(-3.4, -1.4), ylim=(-1.0, 1.0), xticks=[], yticks=[])
    for spine in inset.spines.values():
        spine.set_color("#F0E442"); spine.set_linewidth(2.5)
    ax_holo.indicate_inset_zoom(inset, edgecolor="#F0E442", lw=2)

    # Ground truth and measurement on the same specimen-coordinate grid (the object wave's positions).
    measured_all = magnetic_measured["recorded"]
    true_unfiltered = phase_at(np.stack((obj_y, obj_x), -1))
    cube_outline = (thickness_at(np.stack((obj_y, obj_x), -1)) > 0.5 * side).astype(float)
    ext_nm = (obj_x[0, 0] * 1e9, obj_x[0, -1] * 1e9, obj_y[-1, 0] * 1e9, obj_y[0, 0] * 1e9)
    for ax, data in ((ax_truth, true_unfiltered), (ax_map, np.where(support_summary, measured_all, np.nan))):
        ax.imshow(np.cos(40 * (data - np.nanmean(data[in_cube]))), extent=ext_nm, cmap="gray")
        ax.contour(obj_x * 1e9, obj_y * 1e9, cube_outline, levels=[0.5], colors="#CC79A7", linewidths=3)
        ax.text((cube_x - side / 2) * 1e9, (side / 2) * 1e9 + 3, "cube", color="#CC79A7", fontsize=22,
                fontweight="bold", va="bottom", bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.85})
        ax.set(xlabel="Specimen x (nm)", ylabel="Specimen y (nm)")
    # Crop both maps to just inside the reconstructed (overlap) region so no blank pixels show.
    cols = np.flatnonzero(support_summary.mean(axis=0) > 0.97)
    rows = np.flatnonzero(support_summary[:, cols].all(axis=1))
    x_lo, x_hi = obj_x[0, cols[0]] * 1e9 + 1, obj_x[0, cols[-1]] * 1e9 - 1
    y_lo, y_hi = sorted((obj_y[rows[0], 0] * 1e9, obj_y[rows[-1], 0] * 1e9))
    half = 0.5 * min(x_hi - x_lo, y_hi - y_lo - 2)       # one square window, centred on the cube
    x_mid = 0.5 * (x_lo + x_hi)
    for ax in (ax_truth, ax_map):
        ax.set(xlim=(x_mid - half, x_mid + half), ylim=(-half, half), aspect="equal")
        ax.set_facecolor("black")                        # any remaining blank corner pixels
    for ax in (ax_holo, ax_truth, ax_map):
        ax.set_box_aspect(1)                             # identical square image panels

    row_c = np.argmin(np.abs(cam_y))
    visible = support_summary[row_c]
    x_nm = obj_x[row_c] * 1e9
    ax_profile.plot(x_nm, np.where(visible, centre(true_magnetic)[row_c], np.nan), color="k", lw=3,
                    label="ground truth")
    ax_profile.plot(x_nm, np.where(visible, centre(measured_all)[row_c], np.nan), color="#D55E00", ls="--", lw=2.5,
                    label="measured")
    ax_profile.axvspan((cube_x - side / 2) * 1e9, (cube_x + side / 2) * 1e9, color="#CC79A7", alpha=0.15)
    ax_profile.text(cube_x * 1e9, 0.04, "cube", transform=ax_profile.get_xaxis_transform(), ha="center",
                    color="#CC79A7", fontsize=22, fontweight="bold")
    ax_profile.set(xlabel="Position across the cube (nm)", ylabel="Magnetic phase (rad)")
    ax_profile.legend(frameon=False, fontsize=20, loc="upper left")

    effects_order = ("stray field", "distortion", "on-axis", "off-axis", "coherence")
    names = {"stray field": "stray field\nin reference", "distortion": "projector\ndistortion 1 %",
             "on-axis": "on-axis\naberrations", "off-axis": "off-axis\naberrations",
             "coherence": "partial coherence\n+ shot noise", "all": "all together"}
    values = [budget[key] for key in names]
    colours = [ERROR_COLOURS[key] for key in effects_order] + ["#374151"]
    bars = ax_budget.barh(list(names.values()), [v / signal_m * 100 for v in values], color=colours, height=0.65)
    ax_budget.set_xscale("log")
    for bar, value in zip(bars, values):
        ax_budget.text(bar.get_width() * 1.2, bar.get_y() + bar.get_height() / 2,
                       f"{100 * value / signal_m:.2g} %", va="center", fontsize=22, fontweight="bold")
    ax_budget.invert_yaxis()
    ax_budget.set_xlim(1e-3, 30)
    ax_budget.xaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f"{v:g} %"))
    ax_budget.set(xlabel="Error (% of magnetic signal)")

    titles = ("Microscope column", "Simulated hologram", "Truth", "Measured", "Profile through the cube", "Error budget")
    for ax, letter, title in zip((ax_column, ax_holo, ax_truth, ax_map, ax_profile, ax_budget), "abcdef", titles):
        ax.set_title(f"({letter})  {title}", loc="left", fontweight="bold")
fig.savefig(OUTPUT / "summary_figure.png", dpi=200)
fig.savefig(OUTPUT / "summary_figure.pdf")
plt.show()

## Notes and limits

* **Scale and cost.** Timings are printed above and in the figure title. Propagating every Gaussian through the
  whole column takes milliseconds; rendering the coherent sum on the camera dominates.
* **The wire.** A specimen beamlet is at least about √(λb') wide at the wire, the same scale as the wire's edge
  fringes. The check above measures the resulting error against an exact Fresnel calculation. Set
  `SPLIT_AT_WIRE = True` to split beamlets there; it reduces that error at several times the rendering cost.
* **Specimen model.** The main cube uses the recipe's vector-potential gauge confined to the cube, so it has no
  external stray field; the stray-field section uses a uniformly magnetised magnetite cube with its full field. The 1.5 nm smoothing is part of the specimen model; refine the beamlet waist and smoothing
  together.
* **Distortion.** Projector distortion is a canonical point map at the `IL3` image about the illumination's reference
  wavefront. It displaces the image and fringes as a real distortion does; it does not model the field-dependent
  blur a real projector adds. The Seidel `DistortedLens` kernel is not accurate for micrometre-wide beamlets.
* **Aberrations.** C10 and C12 are exact quadratic Gaussian updates in the back focal plane. C30 = 1.2 mm adds
  little phase at these angles and is included for completeness. The wire edge is a 100 nm sigmoid, below the
  Fresnel scale.
* **Detector.** A 1024² camera with 14 µm pixels replaces the model's 256² Medipix, which would sample the
  fringes at about one pixel each at this magnification.
* **Off-axis aberrations.** The field-dependent objective terms are referenced to the objective aperture (back
  focal plane) and applied with the Seidel kernel at the objective, where beamlets are small enough for its local
  expansion. Their size is chosen for visibility; for a simple objective, scaling with (r/f) suggests much smaller
  values at a 100 nm field, so scale the coefficients to your lens before drawing quantitative conclusions.
* **Not included.** Partial coherence, charging of the biprism wire, detector MTF and noise.